# Palm Vein — Augmentation Pipeline
Steps: **Normalization** | **Rotation augmentation** | **Scale (distance) augmentation**

In [ ]:
import torch
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os

DATASET = r'c:\Users\riyat\OneDrive\Documents\palmvein\palm_vein_ready'
print('PyTorch:', torch.__version__)
print('Dataset path exists:', os.path.isdir(DATASET))
subjects = sorted([d for d in os.listdir(DATASET) if os.path.isdir(os.path.join(DATASET, d))])
print(f'Subjects found: {len(subjects)}')

## Step 1 — Normalization
Converts pixel values from [0, 255] → [0.0, 1.0] then applies ImageNet mean/std normalisation (required for pretrained CNNs).

In [ ]:
# ImageNet stats — used because our backbone was pretrained on ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Load a sample image (grayscale → replicate to 3 channels for pretrained CNN)
sample_path = os.path.join(DATASET, 'subject001', 'sample_000.png')
img_pil = Image.open(sample_path).convert('L')  # grayscale

to_tensor = T.ToTensor()  # [0,255] → [0.0, 1.0]
normalize  = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

img_tensor = to_tensor(img_pil)  # shape: [1, 224, 224]
img_3ch    = img_tensor.repeat(3, 1, 1)  # → [3, 224, 224]
img_norm   = normalize(img_3ch)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_pil, cmap='gray')
axes[0].set_title(f'Original\nrange: [0, 255]')
axes[0].axis('off')
axes[1].imshow(img_norm[0].numpy(), cmap='gray')
axes[1].set_title(f'Normalised (ImageNet)\nrange: [{img_norm.min():.2f}, {img_norm.max():.2f}]')
axes[1].axis('off')
plt.suptitle('Step 1 — Normalisation', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 2 — Rotation Augmentation
Random rotation up to ±180° to handle the fact that palms were captured at different angles across sessions.

In [ ]:
torch.manual_seed(42)

rotation_aug = T.RandomRotation(degrees=180, fill=0)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, subj in enumerate(['subject001', 'subject010', 'subject020', 'subject040', 'subject060']):
    img = Image.open(os.path.join(DATASET, subj, 'sample_002.png')).convert('L')
    aug = rotation_aug(img)
    axes[0, i].imshow(img, cmap='gray'); axes[0, i].set_title(f'{subj}\nOriginal'); axes[0, i].axis('off')
    axes[1, i].imshow(aug, cmap='gray'); axes[1, i].set_title('After rotation'); axes[1, i].axis('off')

plt.suptitle('Step 2 — Rotation Augmentation (random ±180°)', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3 — Scale (Distance) Augmentation
Random zoom between 85%–115% of original size to handle variation in capture distance. Crop is kept at 224×224.

In [ ]:
scale_aug = T.RandomResizedCrop(
    size=224,
    scale=(0.85, 1.15),   # zoom range
    ratio=(0.95, 1.05),   # keep roughly square
    interpolation=T.InterpolationMode.BILINEAR
)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, subj in enumerate(['subject001', 'subject010', 'subject020', 'subject040', 'subject060']):
    img = Image.open(os.path.join(DATASET, subj, 'sample_002.png')).convert('L')
    aug = scale_aug(img)
    axes[0, i].imshow(img, cmap='gray'); axes[0, i].set_title(f'{subj}\nOriginal'); axes[0, i].axis('off')
    axes[1, i].imshow(aug, cmap='gray'); axes[1, i].set_title('After zoom/scale'); axes[1, i].axis('off')

plt.suptitle('Step 3 — Scale Augmentation (random zoom 85%–115%)', fontweight='bold')
plt.tight_layout()
plt.show()

## Full Combined Transform
This is the transform that will be used during **training**. Run this cell to confirm it works end-to-end.

In [ ]:
train_transform = T.Compose([
    T.Grayscale(num_output_channels=3),        # grayscale → 3ch for pretrained CNN
    T.RandomRotation(degrees=180, fill=0),      # Step 2: rotation
    T.RandomResizedCrop(224, scale=(0.85, 1.15), ratio=(0.95, 1.05)),  # Step 3: scale
    T.ColorJitter(brightness=0.1),              # slight brightness variation
    T.ToTensor(),                               # [0,255] → [0.0,1.0]
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),  # Step 1: normalize
])

# Val/test transform — no augmentation, just normalize
val_transform = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Test on one image
img = Image.open(os.path.join(DATASET, 'subject001', 'sample_000.png'))
t   = train_transform(img)
print('Output tensor shape:', t.shape)   # expect torch.Size([3, 224, 224])
print('Value range:         ', round(t.min().item(),3), 'to', round(t.max().item(),3))
print()
print('train_transform ready for training loop.')
print('val_transform   ready for evaluation.')